# Dream interpretations – SQL cleaning notebook

Ce notebook charge `dreams_interpretations.csv` et construit trois tables SQLite :

- `dream_symbols`
- `dream_symbol_aliases`
- `dream_symbol_features`

Le traitement reste piloté par SQL, avec quelques fonctions Python exposées à SQLite pour la normalisation, les slugs, le nettoyage du texte et les regex.

In [1]:
from pathlib import Path
import math
import pandas as pd
import sqlite3
import re
import unicodedata

DATA_PATH = Path('/mnt/data/dreams_interpretations.csv')
DB_PATH = Path('/mnt/data/dreams_interpretations.sqlite')

assert DATA_PATH.exists(), f"Fichier introuvable: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
df.head()

,Dream Symbol,Interpretation
0,Aardvark,To see an aardvark in your dream indicates tha...
1,Abandonment,To dream that you are abandoned suggests that ...
2,Abduction,To dream of being abducted indicates that you ...
3,Aborigine,To see an Aborigine in your dream represents b...
4,Abortion,To dream that you have an abortion suggests th...


In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.isna().sum())

(902, 2)
['Dream Symbol', 'Interpretation']
Dream Symbol      0
Interpretation    0
dtype: int64


In [3]:
def is_nullish(value):
    try:
        return value is None or (isinstance(value, float) and math.isnan(value))
    except Exception:
        return False


def strip_accents(text):
    if is_nullish(text):
        return None
    try:
        text = str(text)
        return ''.join(
            ch for ch in unicodedata.normalize('NFKD', text)
            if not unicodedata.combining(ch)
        )
    except Exception:
        return str(text)


def normalize_label(text):
    if is_nullish(text):
        return None
    try:
        text = strip_accents(text)
        text = text.replace('\xa0', ' ')
        text = text.replace('&', ' and ')
        text = re.sub(r'[^A-Za-z0-9/\\-]+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip().lower()
        return text or None
    except Exception:
        return None


def slugify(text):
    if is_nullish(text):
        return None
    try:
        text = normalize_label(text)
        if text is None:
            return None
        text = re.sub(r'[/\\]+', '-', text)
        text = re.sub(r'[^a-z0-9]+', '-', text)
        text = re.sub(r'-{2,}', '-', text).strip('-')
        return text or None
    except Exception:
        return None


def clean_interpretation(text):
    if is_nullish(text):
        return None
    try:
        text = strip_accents(text)
        text = text.replace('\xa0', ' ')
        text = text.replace('\u200b', ' ')
        text = re.sub(r'\s+', ' ', text).strip()
        return text or None
    except Exception:
        return None


def regexp(pattern, value):
    try:
        if value is None:
            return 0
        return 1 if re.search(pattern, str(value), flags=re.IGNORECASE) else 0
    except Exception:
        return 0

In [4]:
if DB_PATH.exists():
    DB_PATH.unlink()

con = sqlite3.connect(DB_PATH)
con.create_function('NORMALIZE_LABEL', 1, normalize_label)
con.create_function('SLUGIFY', 1, slugify)
con.create_function('CLEAN_INTERPRETATION', 1, clean_interpretation)
con.create_function('REGEXP', 2, regexp)

df.to_sql('dream_interpretations_raw', con, index=False, if_exists='replace')
pd.read_sql_query('SELECT * FROM dream_interpretations_raw LIMIT 5', con)

,Dream Symbol,Interpretation
0,Aardvark,To see an aardvark in your dream indicates tha...
1,Abandonment,To dream that you are abandoned suggests that ...
2,Abduction,To dream of being abducted indicates that you ...
3,Aborigine,To see an Aborigine in your dream represents b...
4,Abortion,To dream that you have an abortion suggests th...


## 1. Table nettoyée `dream_symbols`

In [5]:
con.executescript("""
DROP TABLE IF EXISTS dream_symbols;

CREATE TABLE dream_symbols AS
WITH base AS (
    SELECT
        rowid AS raw_id,
        TRIM("Dream Symbol") AS original_label,
        NORMALIZE_LABEL("Dream Symbol") AS normalized_label,
        SLUGIFY("Dream Symbol") AS slug,
        CLEAN_INTERPRETATION(Interpretation) AS interpretation_text_clean
    FROM dream_interpretations_raw
    WHERE TRIM(COALESCE("Dream Symbol", '')) <> ''
      AND TRIM(COALESCE(Interpretation, '')) <> ''
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY normalized_label, interpretation_text_clean
            ORDER BY raw_id
        ) AS dedupe_rank,
        COUNT(*) OVER (
            PARTITION BY normalized_label, interpretation_text_clean
        ) AS duplicate_count
    FROM base
)
SELECT
    ROW_NUMBER() OVER (ORDER BY normalized_label, original_label, raw_id) AS symbol_id,
    raw_id,
    original_label,
    normalized_label,
    slug,
    interpretation_text_clean,
    duplicate_count,
    CURRENT_TIMESTAMP AS created_at
FROM ranked
WHERE dedupe_rank = 1;

CREATE INDEX idx_dream_symbols_normalized_label ON dream_symbols(normalized_label);
CREATE INDEX idx_dream_symbols_slug ON dream_symbols(slug);
""")

pd.read_sql_query('SELECT * FROM dream_symbols LIMIT 10', con)

,symbol_id,raw_id,original_label,normalized_label,slug,interpretation_text_clean,duplicate_count,created_at
0,1,758,-Turn,-turn,turn,To make a u-turn in your dream indicates that ...,1,2026-03-30 09:04:04
1,2,20,A,a,a,To dream that you are getting acupuncture sugg...,1,2026-03-30 09:04:04
2,3,1,Aardvark,aardvark,aardvark,To see an aardvark in your dream indicates tha...,1,2026-03-30 09:04:04
3,4,2,Abandonment,abandonment,abandonment,To dream that you are abandoned suggests that ...,1,2026-03-30 09:04:04
4,5,3,Abduction,abduction,abduction,To dream of being abducted indicates that you ...,1,2026-03-30 09:04:04
5,6,4,Aborigine,aborigine,aborigine,To see an Aborigine in your dream represents b...,1,2026-03-30 09:04:04
6,7,5,Abortion,abortion,abortion,To dream that you have an abortion suggests th...,1,2026-03-30 09:04:04
7,8,7,Abscess,abscess,abscess,To dream that you have an abscess suggests tha...,1,2026-03-30 09:04:04
8,9,6,Absence,absence,absence,"To dream that someone is absent, especially if...",1,2026-03-30 09:04:04
9,10,8,Abundance,abundance,abundance,To dream of having an abundance of a certain i...,1,2026-03-30 09:04:04


In [6]:
raw_count = pd.read_sql_query('SELECT COUNT(*) AS n FROM dream_interpretations_raw', con)['n'][0]
clean_count = pd.read_sql_query('SELECT COUNT(*) AS n FROM dream_symbols', con)['n'][0]
print('Lignes brutes :', raw_count)
print('Lignes nettoyées :', clean_count)
print('Doublons exacts retirés :', raw_count - clean_count)

Lignes brutes : 902
Lignes nettoyées : 902
Doublons exacts retirés : 0


## 2. Table `dream_symbol_aliases`

In [7]:
con.executescript("""
DROP TABLE IF EXISTS dream_symbol_aliases;

CREATE TABLE dream_symbol_aliases AS
WITH RECURSIVE split(symbol_id, original_label, working_label, piece, rest) AS (
    SELECT
        symbol_id,
        original_label,
        REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/') AS working_label,
        TRIM(CASE
            WHEN INSTR(REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/'), '/') > 0
                THEN SUBSTR(REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/'), 1, INSTR(REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/'), '/') - 1)
            ELSE REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/')
        END) AS piece,
        CASE
            WHEN INSTR(REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/'), '/') > 0
                THEN SUBSTR(REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/'), INSTR(REPLACE(REPLACE(REPLACE(original_label, ' / ', '/'), '/ ', '/'), ' /', '/'), '/') + 1)
            ELSE ''
        END AS rest
    FROM dream_symbols

    UNION ALL

    SELECT
        symbol_id,
        original_label,
        working_label,
        TRIM(CASE
            WHEN INSTR(rest, '/') > 0 THEN SUBSTR(rest, 1, INSTR(rest, '/') - 1)
            ELSE rest
        END) AS piece,
        CASE
            WHEN INSTR(rest, '/') > 0 THEN SUBSTR(rest, INSTR(rest, '/') + 1)
            ELSE ''
        END AS rest
    FROM split
    WHERE rest <> ''
),
all_aliases AS (
    SELECT symbol_id, original_label AS alias_label, 'original' AS alias_type
    FROM dream_symbols

    UNION ALL

    SELECT symbol_id, piece AS alias_label, 'slash_variant' AS alias_type
    FROM split
    WHERE working_label LIKE '%/%'
      AND TRIM(piece) <> ''
)
SELECT DISTINCT
    ROW_NUMBER() OVER (ORDER BY symbol_id, alias_type, alias_label) AS alias_id,
    symbol_id,
    alias_label,
    NORMALIZE_LABEL(alias_label) AS alias_normalized,
    SLUGIFY(alias_label) AS alias_slug,
    alias_type
FROM all_aliases
WHERE TRIM(alias_label) <> '';

CREATE INDEX idx_dream_symbol_aliases_symbol_id ON dream_symbol_aliases(symbol_id);
CREATE INDEX idx_dream_symbol_aliases_alias_normalized ON dream_symbol_aliases(alias_normalized);
""")

pd.read_sql_query("SELECT * FROM dream_symbol_aliases WHERE alias_type = 'slash_variant' LIMIT 20", con)

,alias_id,symbol_id,alias_label,alias_normalized,alias_slug,alias_type
0,22,21,Actor,actor,actor,slash_variant
1,23,21,Actress,actress,actress,slash_variant
2,835,832,Waiter,waiter,waiter,slash_variant
3,836,832,Waitress,waitress,waitress,slash_variant


## 3. Table `dream_symbol_features`

In [8]:
con.executescript("""
DROP TABLE IF EXISTS dream_symbol_features;

CREATE TABLE dream_symbol_features AS
WITH feature_base AS (
    SELECT
        symbol_id,
        original_label,
        interpretation_text_clean,
        REGEXP('\\b(fear|afraid|terror|panic|anxiety|anxious|scared|fright|betray|threat|danger|abandon(?:ed|ment)?)\\b', interpretation_text_clean) AS fear,
        REGEXP('\\b(love|relationship|partner|marriage|married|family|mother|father|child|children|friend|friends|lover|romance|betrayal)\\b', interpretation_text_clean) AS relationships,
        REGEXP('\\b(work|job|career|business|profession|office|school|success|achievement|promotion|goal|ambition)\\b', interpretation_text_clean) AS career,
        REGEXP('\\b(change|transform|transformation|growth|rebirth|renewal|evolve|transition|healing|new direction)\\b', interpretation_text_clean) AS transformation,
        REGEXP('\\b(spiritual|spirituality|soul|spirit|divine|sacred|faith|intuition|inner self|higher self|meditation)\\b', interpretation_text_clean) AS spirituality,
        REGEXP('\\b(health|healthy|illness|disease|body|well-being|wellbeing|injury|wound|healing|stress|pregnan|abortion)\\b', interpretation_text_clean) AS health,
        REGEXP('\\b(happy|joy|peace|calm|confidence|hope|optimism|love|freedom|relief)\\b', interpretation_text_clean) AS positive_signal,
        REGEXP('\\b(fear|sad|grief|anger|anxiety|stress|loss|betray|pain|neglect|trauma|conflict)\\b', interpretation_text_clean) AS negative_signal
    FROM dream_symbols
)
SELECT
    symbol_id,
    original_label,
    fear,
    relationships,
    career,
    transformation,
    spirituality,
    health,
    CASE
        WHEN positive_signal = 1 AND negative_signal = 1 THEN 'mixed'
        WHEN negative_signal = 1 THEN 'negative'
        WHEN positive_signal = 1 THEN 'positive'
        ELSE 'neutral'
    END AS emotional_tone
FROM feature_base;

CREATE INDEX idx_dream_symbol_features_symbol_id ON dream_symbol_features(symbol_id);
CREATE INDEX idx_dream_symbol_features_emotional_tone ON dream_symbol_features(emotional_tone);
""")

pd.read_sql_query('SELECT * FROM dream_symbol_features LIMIT 10', con)

,symbol_id,original_label,fear,relationships,career,transformation,spirituality,health,emotional_tone
0,1,-Turn,1,1,1,1,1,0,mixed
1,2,A,0,0,0,1,0,1,neutral
2,3,Aardvark,0,0,1,0,0,0,neutral
3,4,Abandonment,1,0,0,1,0,1,negative
4,5,Abduction,0,0,0,0,0,0,neutral
5,6,Aborigine,0,0,0,0,1,1,neutral
6,7,Abortion,1,0,0,1,0,1,negative
7,8,Abscess,0,0,0,0,0,0,neutral
8,9,Absence,0,0,0,0,0,0,neutral
9,10,Abundance,0,0,0,0,0,0,neutral


## 4. Contrôles rapides

In [9]:
pd.read_sql_query("SELECT emotional_tone, COUNT(*) AS n FROM dream_symbol_features GROUP BY emotional_tone ORDER BY n DESC", con)

,emotional_tone,n
0,neutral,758
1,negative,73
2,positive,55
3,mixed,16


In [10]:
pd.read_sql_query("""
SELECT
    SUM(fear) AS fear,
    SUM(relationships) AS relationships,
    SUM(career) AS career,
    SUM(transformation) AS transformation,
    SUM(spirituality) AS spirituality,
    SUM(health) AS health
FROM dream_symbol_features
""", con)

,fear,relationships,career,transformation,spirituality,health
0,57,115,77,74,57,48


In [11]:
user_input = 'actress'

pd.read_sql_query("""
SELECT
    s.symbol_id,
    s.original_label,
    s.interpretation_text_clean,
    f.emotional_tone,
    f.fear,
    f.relationships,
    f.career,
    f.transformation,
    f.spirituality,
    f.health
FROM dream_symbol_aliases a
JOIN dream_symbols s ON s.symbol_id = a.symbol_id
LEFT JOIN dream_symbol_features f ON f.symbol_id = s.symbol_id
WHERE a.alias_normalized = NORMALIZE_LABEL(?)
ORDER BY s.original_label
""", con, params=[user_input])

,symbol_id,original_label,interpretation_text_clean,emotional_tone,fear,relationships,career,transformation,spirituality,health
0,21,Actor/Actress,To see an actor or actress in your dream repre...,neutral,0,0,0,0,0,0


In [12]:
pd.read_sql_query('SELECT * FROM dream_symbols', con).to_csv('/mnt/data/dream_symbols_clean.csv', index=False)
pd.read_sql_query('SELECT * FROM dream_symbol_aliases', con).to_csv('/mnt/data/dream_symbol_aliases.csv', index=False)
pd.read_sql_query('SELECT * FROM dream_symbol_features', con).to_csv('/mnt/data/dream_symbol_features.csv', index=False)

print('Notebook prêt')
print('Base SQLite :', DB_PATH)

Notebook prêt
Base SQLite : /mnt/data/dreams_interpretations.sqlite
